# OcuWeave · annotation review and controlled development
Marchel Andrian Shevchenko · Data Sorcerers · 2026 · software v0.4.0

This notebook reviews development image/mask geometry, registers nine controlled runs, and executes them only when CUDA is available. Results are **pending until measured**. It preserves the old test results and does not load test images. The experimental local-detail branch is independently implemented without U-Net or pretrained weights; no gain or novelty is asserted.

All records stay in the existing private Drive project. CPU review can run while Colab GPU quota is unavailable. No paid compute purchase is required or initiated. See the repository results register for the completed v0.2.0 pilot and v0.3.0 quality audit.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys, os, json, hashlib, shutil
from IPython.display import display, Image, Markdown
drive.mount('/content/drive')
PROJECT=Path('/content/drive/MyDrive/OcuWeave — Datasets and experiments')
assert PROJECT.is_dir()
RELEASE='v0.4.0'
REPO=Path('/content/ocuweave-development-'+RELEASE)
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--branch',RELEASE,'https://github.com/chenko-the-sorcerers/ocuweave.git',str(REPO)],check=True,capture_output=True)
assert subprocess.check_output(['git','describe','--tags','--exact-match'],cwd=REPO,text=True).strip()==RELEASE
assert not subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip()
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT=PROJECT/'02 — Audits/development-v0.4.0'
LOGS=OUTPUT/'logs'/STAMP
LOGS.mkdir(parents=True,exist_ok=False)
def run(args,name):
    result=subprocess.run(args,cwd=REPO,capture_output=True,text=True)
    (LOGS/name).write_text(result.stdout+result.stderr)
    print(name,'exit:',result.returncode,(result.stdout+result.stderr)[-2200:])
    result.check_returncode()
run([sys.executable,'-m','pip','install','--no-deps','-e','.'],'install.log')
import torch
print({'source_commit':REVISION,'torch':str(torch.__version__),'cuda_available':torch.cuda.is_available()})


## 1. Verify software
Run synthetic geometry, gradient, native interoperability and end-to-end development checks before reading real images.

In [ ]:
run(['cmake','-S','.','-B','build','-DBUILD_TESTING=ON'],'cmake.log')
run(['cmake','--build','build'],'native-build.log')
run(['ctest','--test-dir','build','--output-on-failure'],'ctest.log')
os.environ['OCUWEAVE_NATIVE_LIBRARY']=str(REPO/'build/libocuweave_native.so')
run([sys.executable,'-m','unittest','discover','-s','tests','-v'],'tests.log')
run([sys.executable,'scripts/audit_data.py','examples/synthetic_manifest.json'],'synthetic-audit.log')


## 2. Verify frozen inputs
Copy only the OTFID archive to temporary Colab storage and verify its receipt. Preserve the existing split and the two saved test-result files. No test predictions are recomputed.

In [ ]:
def digest(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
SOURCE=PROJECT/'01 — Raw datasets/OTFID_v3'
ARCHIVE=Path('/content/ocuweave-quality-otfid.zip')
if not ARCHIVE.exists():shutil.copyfile(SOURCE/'Ocular_Toxoplasmosis_Data_V3.zip',ARCHIVE)
assert digest(ARCHIVE)==json.loads((SOURCE/'download_manifest.json').read_text())['sha256']
MANIFEST=PROJECT/'02 — Audits/otfid-v3-binary-v2-seed42-v0.2.0/manifest.json'
assert digest(MANIFEST)=='b8cc4f42779d57ad5ad0fa18704730edaf62650d5594996ffab1cc6d25cb980f'
PILOT=PROJECT/'04 — Runs/v0.2.0/pilot-seed42-20epochs'
TEST_HASHES={'query-test.json':'42a1a5aac443816cb4ed08c87411249f03ef8e283851674c389ca650b90d8f42','fcn-test.json':'0a94ce5aa157d0600ea8ea41a24797e4f3d3b38a8c3887877145f8dc06191f2c'}
assert all(digest(PILOT/name)==value for name,value in TEST_HASHES.items())
QUALITY=PROJECT/'02 — Audits/quality-review-v0.3.0'
prior=json.loads((QUALITY/'otfid-query-validation/diagnostics.json').read_text())
assert prior['manifest_sha256']==digest(MANIFEST)
SELECTED=list(dict.fromkeys(sum(prior['review_selection'].values(),[])))
print('Frozen archive, manifest and saved test hashes verified. Selected development examples:',len(SELECTED))


## 3. Annotation and preprocessing review
Check all development pairs and compare 192/384 nearest-neighbor roundtrips on the original grid. These metrics describe resampling distortion, not model performance. Clinical annotation correctness remains unadjudicated.

In [ ]:
REVIEW=OUTPUT/'annotation-geometry'
if not REVIEW.exists():
    run([sys.executable,'scripts/review_annotations.py','--manifest',str(MANIFEST),'--archive',str(ARCHIVE),'--output',str(REVIEW),'--selected',*SELECTED,'--allow-unverified-patients'],'annotation-review.log')
assert (REVIEW/'summary.json').exists(),'Incomplete review; preserve it and investigate'
review=json.loads((REVIEW/'summary.json').read_text())
assert review['manifest_sha256']==digest(MANIFEST) and review['archive_sha256']==digest(ARCHIVE)
assert not review['pair_failures'],'Pair mismatch blocks training'
print(json.dumps(review,indent=2))


## 4. Inspect boundary overlays
Cyan contours are reference-mask boundaries after the stated transform. They are not predictions. All selected panels remain in the audit folder; two are displayed here for readable comparison.

In [ ]:
panels=sorted(REVIEW.glob('overlay-*.png'))
assert panels
for path in panels[:2]:display(Image(filename=str(path),width=1000))
display(Markdown('Technical consistency is not clinical boundary approval. Record expert adjudication separately; no annotation is automatically edited.'))


## 5. Register and run controlled experiments
Query 192 vs query 384 tests resolution; query 384 vs detail 384 tests the new branch. Seeds: 42, 123, 2026. Each run uses 20 epochs, batch 8, 500 updates for the frozen 196-image training set, unchanged loss/optimizer/split. GPU OOM or interruption stops the suite and preserves incomplete files; do not silently lower the budget.

Each run gets a separate process, log, history, checkpoint, original-grid validation, memory/time measurements and a completion receipt. Existing completed runs are verified before reuse. Training is skipped when CUDA is unavailable; skipped results are not zero scores.

In [ ]:
SUITE=PROJECT/'04 — Runs/v0.4.0/resolution-detail-seeds'
args=[sys.executable,'scripts/run_ablations.py','--manifest',str(MANIFEST),'--archive',str(ARCHIVE),'--output',str(SUITE)]
run(args,'register-suite.log')
if torch.cuda.is_available():
    run(args+['--execute','--device','cuda'],'execute-suite.log')
    training_status='See per-run status; CUDA execution requested'
else:
    training_status='Not executed: this Colab session has no CUDA GPU. Registered runs remain pending.'
print(training_status)


## 6. Results and interpretation
Only complete, hash-verified runs enter comparison tables. Original-grid validation differs from the historical 192-pixel pilot metric. Paired seed differences describe initialization variability, not patient-level uncertainty.

In [ ]:
register=json.loads((SUITE/'summary.json').read_text())
display(Markdown((SUITE/'README.md').read_text()))
print(json.dumps(register['paired_comparisons'],indent=2))
completed=[r for r in register['runs'] if r['status']=='completed']
if completed:
    import matplotlib.pyplot as plt
    fig,ax=plt.subplots(figsize=(10,4),layout='constrained')
    for name,color in [('query192','#2457A7'),('query384','#007F73'),('detail384','#AC5518')]:
        subset=[r for r in completed if r['run'].startswith(name+'-seed')]
        ax.scatter([r['seed'] for r in subset],[r['positive_dice'] for r in subset],label=name,color=color)
    ax.set(xlabel='Initialization seed',ylabel='Original-grid validation positive Dice',ylim=(0,1),title='Completed development runs; no test tuning')
    ax.legend();fig.savefig(SUITE/'completed-runs.png',dpi=150);display(fig);plt.close(fig)
else:
    display(Markdown('**No new training result is available.** Annotation diagnostics and the predeclared experiment register are the completed outputs of this session.'))


## 7. Archive every result and preserve the test record
Save software provenance, the full documentation snapshot and a SHA-256 inventory of private reports, logs, figures, checkpoints and notebooks. FairVLMed multimodal training remains gated on semantic/temporal leakage review and separate patient pairing.

In [ ]:
assert all(digest(PILOT/name)==value for name,value in TEST_HASHES.items())
provenance={'software':RELEASE,'source_commit':REVISION,'manifest_sha256':digest(MANIFEST),'archive_sha256':digest(ARCHIVE),'preserved_test_sha256':TEST_HASHES,'training_status':training_status,'completed_runs':len(completed),'planned_runs':len(register['runs']),'clinical_boundary_review_complete':False,'emr_training_cleared':False}
(LOGS/'session-provenance.json').write_text(json.dumps(provenance,indent=2))
SNAPSHOT=PROJECT/'05 — Research documentation'/RELEASE/STAMP
run([sys.executable,'scripts/archive_research_record.py','--project',str(PROJECT),'--output',str(SNAPSHOT)],'documentation-snapshot.log')
print('Documentation snapshot:',SNAPSHOT)
print('Original test results unchanged. No test images evaluated.')
